# TC3: 3x3 양자컴퓨터 실행 — PADO 방식 vs standard 방식 (parameter-shift)

케이스 하나(`configs/TC3.json`: 3x3 인스턴스 0, 상수 초기화 c = 0.1)에서 두 방식을 양자컴퓨터로 학습한다.

| 방식 | 시작점 | 양자컴퓨터 학습 |
|---|---|---|
| PADO (`pado_safe`) | PADO LWPP 사전학습 → 증류(\|θ\| ≤ 0.3 제거) 뒤의 θ, 증류된 회로 | parameter-shift 5 step |
| standard (`standard`) | 상수 초기값 θ_init, 전체 회로 | parameter-shift 20 step |

노트북 흐름: 설정 → (API 키) → backend → 1. 문제와 시작점 → 2. 회로 → 3. 큐비트 배치와 transpile → 4. parameter-shift 학습 → 5. 증류 전 측정 → 6. 끝

- 설정은 `configs/TC3.json`에서 읽는다. 시작점은 `prepare_start_points.py`가 GPU에서 미리 만든 `results_on_<GPU>/TC3/<케이스>/start_points/`.
- 결과는 `results_on_<backend 이름>/TC3/<케이스>/<방식>/`에 step마다 저장된다. 중간에 끊기면 학습 셀을 다시 실행하면 이어서 돌고, 끝난 방식은 건너뛴다.
- 지표·판정·그림은 이 노트북이 아니라 `python metric.py --exp TC3`가 만든다.

**실기기로 바꾸는 법**: `configs/TC3.json`의 `qpu.backend_mode`를 `"ibm"`, `qpu.ibm_backend`에 기기 이름을 넣는다. API 키는 아래 키 셀에 넣어 실행한 뒤 지운다. 비워 두면 이 컴퓨터에 저장된 IBM 계정(`~/.qiskit/qiskit-ibm.json`)을 쓴다.

## 설정 (`configs/TC3.json`)

In [1]:
import datetime
import importlib
import json
import time
import warnings
from pathlib import Path

import numpy as np
import rustworkx as rx
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import EstimatorV2, QiskitRuntimeService

from common import device_tag, save_json, save_qasm   # quantum/common.py

REPO = Path.cwd().parent          # 이 노트북은 quantum/ 폴더에서 실행한다
cfg = json.loads((REPO / "configs" / "TC3.json").read_text())
q = cfg["qpu"]
BACKEND_MODE, FAKE_BACKEND, IBM_BACKEND = q["backend_mode"], q["fake_backend"], q["ibm_backend"]
SHOTS, STEPS, LR = q["shots"], q["steps"], cfg["lr"]                     # 회로당 shots, 방식별 step 수, Adam 학습률
RESILIENCE_LEVEL, REP_DELAY = q["resilience_level"], q["rep_delay"]       # 실기기 오차 완화 수준, shot 사이 대기(초, None = 기기 기본값)
OPT_LEVEL, SEED_TRANSPILER, TWO_Q_GATE = q["optimization_level"], q["seed_transpiler"], q["two_q_gate"]
CASE = f"{cfg['grid']}_inst{cfg['instance']}_seed{cfg['seed_base'] + cfg['instance']}"
START_DIR = REPO / f"results_on_{device_tag()}" / cfg["name"] / CASE / "start_points"   # 두 방식의 시작점 (GPU에서 준비)
print(f"{cfg['name']} {CASE} | {BACKEND_MODE} {IBM_BACKEND if BACKEND_MODE == 'ibm' else FAKE_BACKEND} | shots {SHOTS} | steps {STEPS}")

TC3 grid3x3_inst0_seed120490 | fake FakeNighthawk | shots 1024 | steps {'pado_safe': 5, 'standard': 20}


## API 키 (실기기에서만)

`qpu.backend_mode = "ibm"`일 때만 쓰인다. 키를 넣고 아래 backend 셀까지 실행한 뒤 이 셀의 키는 지운다. 키는 파일에 저장하지 않는다.

In [2]:
TOKEN = ""     # IBM Cloud API 키. 비우면 이 컴퓨터에 저장된 계정을 쓴다. 입력했으면 실행 뒤 지운다
INSTANCE = ""  # 인스턴스 CRN 또는 이름 (비우면 계정 기본값)

## backend

`FakeNighthawk`는 qiskit-ibm-runtime에 들어 있는 Nighthawk(정사각 격자 연결) 스냅샷이다. 패키지 스스로 "이 fake backend의 오류 값은 실제 기기의 일반적인 오류 값을 나타내도록 만든 것이 아니다"라고 경고하므로, fake 결과의 노이즈 크기는 실기기 예측으로 쓰지 않는다. 연결 구조와 transpile 결과, 코드 경로 검증용이다. 기기 정보(캘리브레이션 일자, rep_delay, 설정)는 `backend.json`에 저장한다.

In [3]:
warnings.filterwarnings("ignore", message="Properties of fake_")   # fake backend의 "오류 값이 실기기를 대표하지 않는다" 경고 (위 설명 참고)
if BACKEND_MODE == "ibm":
    if TOKEN:
        service = QiskitRuntimeService(channel="ibm_quantum_platform", token=TOKEN, instance=INSTANCE or None)
    else:
        service = QiskitRuntimeService()        # 이 컴퓨터에 저장된 계정
    backend = service.backend(IBM_BACKEND)
else:
    service = None
    backend = getattr(importlib.import_module("qiskit_ibm_runtime.fake_provider"), FAKE_BACKEND)()   # 기기 스냅샷(연결 구조 + 노이즈 모델)

cfg_b = backend.configuration()
rep = REP_DELAY if REP_DELAY is not None else cfg_b.default_rep_delay
if REP_DELAY is not None:
    assert cfg_b.rep_delay_range[0] <= REP_DELAY <= cfg_b.rep_delay_range[1], "rep_delay가 기기 허용 범위 밖"
try:
    calibration = str(backend.properties().last_update_date)
except Exception:
    calibration = None

OUT_ROOT = REPO / f"results_on_{backend.name}" / cfg["name"] / CASE   # 결과 폴더 이름에 기기 이름이 들어간다
OUT_ROOT.mkdir(parents=True, exist_ok=True)
backend_info = {"backend": backend.name, "mode": BACKEND_MODE, "num_qubits": backend.num_qubits,
                "processor_type": getattr(cfg_b, "processor_type", None), "calibration_date": calibration,
                "default_rep_delay_s": cfg_b.default_rep_delay, "rep_delay_range_s": list(cfg_b.rep_delay_range), "rep_delay_used_s": rep,
                "shots": SHOTS, "steps": STEPS, "resilience_level": RESILIENCE_LEVEL if BACKEND_MODE == "ibm" else None,
                "optimization_level": OPT_LEVEL, "seed_transpiler": SEED_TRANSPILER,
                "recorded_at": datetime.datetime.now().isoformat(timespec="seconds")}
save_json(OUT_ROOT / "backend.json", backend_info)
print(backend.name, backend.num_qubits, "qubits |", backend_info["processor_type"], "| 캘리브레이션", calibration)
print(f"rep_delay: 기본 {cfg_b.default_rep_delay * 1e6:.0f} µs, 허용 {cfg_b.rep_delay_range[0] * 1e6:.0f}~{cfg_b.rep_delay_range[1] * 1e6:.0f} µs, 이번 설정 {rep * 1e6:.0f} µs")
print("결과 폴더:", OUT_ROOT.relative_to(REPO))

fake_nighthawk 120 qubits | {'family': 'Nighthawk', 'revision': '1'} | 캘리브레이션 2025-12-08 09:37:09-05:00
rep_delay: 기본 250 µs, 허용 20~500 µs, 이번 설정 250 µs
결과 폴더: results_on_fake_nighthawk/TC3/grid3x3_inst0_seed120490


## 1. 문제와 시작점

In [4]:
# 시작점 파일 두 개: circuit.json(게이트 목록), params.json(θ 값, brute force E_min/E_max, 노이즈 없는 에너지)
circ_json = json.loads((START_DIR / "circuit.json").read_text())
start = json.loads((START_DIR / "params.json").read_text())
n, e_min, e_max = circ_json["n_qubits"], start["e_min"], start["e_max"]
ar = lambda e: (e_max - e) / (e_max - e_min)   # approximation ratio: E_max이면 0, 바닥 상태 E_min이면 1

# 방식별 회로와 시작 θ
#   standard : 전체 회로(각도 114개)와 상수 초기값
#   pado_safe: 증류 후 남은 게이트만 있는 회로(각도 61개)와 사전학습·증류를 거친 θ
gates = {"standard": circ_json["gates"], "pado_safe": circ_json["pruned_gates"]}
theta0 = {"standard": np.array(start["theta_init"]), "pado_safe": np.array(start["theta_pado_safe_start"])}
e_start = {"standard": start["exact_energy"]["standard_start"], "pado_safe": start["exact_energy"]["pado_safe_start"]}
for run in ["pado_safe", "standard"]:
    print(f"{run:10s} angles={len(theta0[run]):3d}  circuits/step={2 * len(theta0[run]) + 1:3d}  "
          f"noiseless start AR={ar(e_start[run]):.4f}")
print(f"brute force: E_min={e_min:.6f}, E_max={e_max:.6f}")

pado_safe  angles= 61  circuits/step=123  noiseless start AR=0.9311
standard   angles=114  circuits/step=229  noiseless start AR=0.5291
brute force: E_min=-8.656307, E_max=9.354969


## 2. 회로와 관측량

게이트 순서는 `circuit.json`과 같다. 각도 규약 R_P(θ) = exp(−iθP/2)는 Qiskit의 `rzz`, `rz`, `rx`와 같다. 파라미터는 `ParameterVector`로 만들어 행렬의 열 순서가 θ 번호와 같게 한다(이름이 `t10`, `t2`처럼 정렬되는 것을 피함). 마지막 줄은 노이즈 없는 Qiskit 에너지가 PennyLane으로 계산해 둔 시작점 에너지와 같은지 확인한다.

In [5]:
# 관측량 H = Σ J_ij Z_i Z_j + Σ h_i Z_i  (문제 해밀토니안)
H = SparsePauliOp.from_sparse_list([("ZZ", [u, v], w) for u, v, w in circ_json["edges"]] +
                                   [("Z", [q], h) for q, h in circ_json["fields"]], num_qubits=n)

# 방식별 파라미터 회로: circuit.json의 게이트를 순서대로 Qiskit 게이트로 옮긴다
circuits = {}
for run in ["pado_safe", "standard"]:
    theta = ParameterVector("theta", len(theta0[run]))   # theta[0], theta[1], ... 번호 순서가 θ 배열 순서와 같다
    qc = QuantumCircuit(n)
    for g in gates[run]:
        if g["gate"] == "H":
            qc.h(g["wires"][0])
        elif g["gate"] == "RZZ":                         # exp(-i θ ZZ / 2)
            qc.rzz(theta[g["param"]], *g["wires"])
        elif g["gate"] == "RZ":                          # exp(-i θ Z / 2)
            qc.rz(theta[g["param"]], g["wires"][0])
        elif g["gate"] == "RX":                          # exp(-i θ X / 2)
            qc.rx(theta[g["param"]], g["wires"][0])
    circuits[run] = qc
    # 확인: 노이즈 없는 Qiskit 에너지 == 시작점 파일의 에너지(PennyLane으로 계산)
    e = float(StatevectorEstimator().run([(qc, H, theta0[run])]).result()[0].data.evs)
    assert abs(e - e_start[run]) < 1e-8, (run, e, e_start[run])
    print(f"{run:10s} noiseless E(theta0) = {e:+.10f}  (start_points: {e_start[run]:+.10f})")

pado_safe  noiseless E(theta0) = -7.4149925623  (start_points: -7.4149925623)
standard   noiseless E(theta0) = -0.1747624990  (start_points: -0.1747624990)


## 3. 큐비트 배치와 transpile

### 3-1. 큐비트 배치: 3x3 격자
transpile에 배치를 맡기면 논리 큐비트 9개가 기기의 아무 물리 큐비트에나 놓일 수 있다. 그러면 격자에서 이웃인 두 큐비트가 기기에서는 멀리 떨어져 SWAP이 늘어날 수 있다. 그래서 문제의 3x3 격자(최근접 이웃 연결 12개)가 기기의 연결 구조 안에 **그대로** 들어가는 물리 큐비트 배치를 직접 고른다.

1. 논리 격자 그래프: 큐비트 q = 3·i + j (i행 j열), 오른쪽·아래쪽 이웃과 연결 (12개).
2. 기기의 연결 그래프 안에서 이 격자가 부분 그래프로 들어가는 모든 배치를 찾는다 (`rustworkx.vf2_mapping`).
3. 배치마다 점수 = 격자 연결 12쌍의 2큐비트 게이트 오류 합 + 9큐비트의 측정 오류 합. 가장 작은 배치를 쓴다 (오류 값은 기기의 최신 보정 데이터 `backend.target`).
4. 두 방식 모두 같은 배치를 쓴다.

대각선(NNN) 연결 8개는 정사각 격자 기기에서도 바로 연결돼 있지 않으므로 transpile이 SWAP을 넣는다. heavy-hex 기기(예: FakeFez)에는 3x3 격자가 그대로 들어가는 배치가 없다. 그 경우에는 그렇게 출력하고 transpiler의 자동 배치를 쓴다.

In [6]:
# 1. 논리 3x3 격자의 최근접 이웃 연결
grid = rx.PyGraph()
grid.add_nodes_from(range(n))
for i in range(3):
    for j in range(3):
        if j + 1 < 3:
            grid.add_edge(3 * i + j, 3 * i + j + 1, None)       # 오른쪽 이웃
        if i + 1 < 3:
            grid.add_edge(3 * i + j, 3 * (i + 1) + j, None)     # 아래쪽 이웃

# 2. 기기 연결 그래프 안에 격자가 그대로 들어가는 모든 배치, 3. 오류 점수
coupling = backend.coupling_map.graph.to_undirected(multigraph=False)
twoq_err = backend.target[TWO_Q_GATE]
candidates = []
for mapping in rx.vf2_mapping(coupling, grid, subgraph=True, induced=False):   # {물리 큐비트: 논리 큐비트}
    layout = [0] * n
    for phys, logical in mapping.items():
        layout[logical] = phys                                  # layout[논리 큐비트] = 물리 큐비트
    score = 0.0
    for u, v in grid.edge_list():
        props = twoq_err.get((layout[u], layout[v])) or twoq_err.get((layout[v], layout[u]))
        score += props.error if props is not None and props.error is not None else 1.0
    for qb in range(n):
        props = backend.target["measure"].get((layout[qb],))
        score += props.error if props is not None and props.error is not None else 1.0
    candidates.append((score, layout))

if candidates:
    score, LAYOUT = min(candidates)
    print(f"3x3 격자 배치 {len(candidates)}개 중 오류 점수가 가장 작은 배치 (점수 {score:.4f})")
    print("물리 큐비트 번호를 논리 격자 모양으로 (행 i, 열 j):")
    for i in range(3):
        print("   " + "  ".join(f"{LAYOUT[3 * i + j]:4d}" for j in range(3)))
else:
    LAYOUT = None
    print(f"{backend.name}에는 3x3 격자가 그대로 들어가는 배치가 없다 (heavy-hex 등). transpiler 자동 배치를 쓴다.")

3x3 격자 배치 640개 중 오류 점수가 가장 작은 배치 (점수 0.0682)
물리 큐비트 번호를 논리 격자 모양으로 (행 i, 열 j):
     27    28    29
     37    38    39
     47    48    49


### 3-2. transpile

양자컴퓨터는 정해진 기본 게이트(Nighthawk는 CZ, RZ, SX, X)와 연결된 큐비트 쌍에서만 2큐비트 게이트를 실행할 수 있다. transpile은 위 회로를 그 조건에 맞는 회로(ISA 회로)로 바꾼다. `initial_layout=LAYOUT`으로 3-1에서 고른 배치를 고정하고, 연결되지 않은 쌍(대각선 NNN)만 SWAP으로 잇는다. 파라미터는 비워 둔 채로 한 번만 transpile하고, 학습 중에는 값만 바꿔 넣는다. 관측량 H도 회로가 놓인 물리 큐비트 번호에 맞게 옮긴다(`apply_layout`). 배치와 회로 크기는 `backend.json`에 기록한다.

In [7]:
pm = generate_preset_pass_manager(optimization_level=OPT_LEVEL, backend=backend, initial_layout=LAYOUT,
                                  seed_transpiler=SEED_TRANSPILER)
isa, isa_H, phys = {}, {}, {}
backend_info["layout_grid_fixed"] = LAYOUT is not None
backend_info["isa"] = {}
for run in ["pado_safe", "standard"]:
    isa[run] = pm.run(circuits[run])                      # 기기용 회로 (파라미터는 비어 있음)
    isa_H[run] = H.apply_layout(isa[run].layout)          # 관측량도 같은 물리 큐비트로
    phys[run] = isa[run].layout.initial_index_layout(filter_ancillas=True)
    if LAYOUT is not None:
        assert phys[run] == LAYOUT, (run, phys[run], LAYOUT)   # 고른 격자 배치가 그대로 쓰였는지 확인
    two_q = sum(1 for inst in isa[run].data if inst.operation.num_qubits == 2)
    backend_info["isa"][run] = {"physical_qubits": phys[run], "two_qubit_gates": two_q, "depth": isa[run].depth(),
                                "two_qubit_depth": isa[run].depth(lambda inst: inst.operation.num_qubits == 2)}
    print(f"{run:10s} 2-qubit gates={two_q}  depth={isa[run].depth()}  2-qubit depth={backend_info['isa'][run]['two_qubit_depth']}  "
          f"physical qubits={phys[run]}")
save_json(OUT_ROOT / "backend.json", backend_info)

pado_safe  2-qubit gates=32  depth=73  2-qubit depth=15  physical qubits=[27, 28, 29, 37, 38, 39, 47, 48, 49]
standard   2-qubit gates=156  depth=309  2-qubit depth=83  physical qubits=[27, 28, 29, 37, 38, 39, 47, 48, 49]


## 4. parameter-shift 학습

### parameter-shift란
이 회로의 게이트는 모두 R_P(θ) = exp(−iθP/2) 꼴이다(P는 Z⊗Z, Z, X). 이런 게이트 하나의 각도 θ_i만 바꾸면 에너지 E는 θ_i에 대한 사인 곡선이 된다. 그래서 θ_i 방향의 기울기를 근사 없이 두 번의 측정으로 구할 수 있다.

  **g_i = ∂E/∂θ_i = ( E(θ + π/2·e_i) − E(θ − π/2·e_i) ) / 2**

여기서 e_i는 i번째 성분만 1인 벡터다(번호는 코드와 같이 0부터 센다). 즉 파라미터마다 "θ_i만 +π/2 옮긴 회로"와 "θ_i만 −π/2 옮긴 회로"를 한 번씩 돌려, 두 에너지의 차를 2로 나눈다.

### job 하나 = step 하나
한 step에 필요한 회로를 모두 한 job에 담아 보낸다. 실제로 보내는 것은 파라미터가 비어 있는 회로 1개, 관측량 H, 그리고 아래 파라미터 행렬이다. 이 셋을 묶은 것을 Qiskit에서는 PUB(primitive unified bloc, Estimator에 넣는 입력 단위)라고 한다.

```
estimator.run([ (ISA 회로, H, rows) ])        ← job 1개

rows (2N+1 행 × N 열), N = 파라미터 수 (PADO 61, standard 114), 파라미터 번호 i = 0 .. N−1
  행 0      : θ                     → E(θ)       이 step의 에너지 (AR 기록용)
  행 1      : θ + π/2·e_0           → E_0+
  행 2      : θ − π/2·e_0           → E_0−
  ...
  행 2i+1   : θ + π/2·e_i           → E_i+
  행 2i+2   : θ − π/2·e_i           → E_i−
  ...
  행 2N     : θ − π/2·e_(N−1)       → E_(N−1)−

기울기  g_i = (E_i+ − E_i−) / 2        →  Adam으로 θ 업데이트  →  다음 step의 job
```

해밀토니안의 항(ZZ, Z)은 모두 Z 기저에서 함께 측정되므로, Estimator는 행마다 회로 1개를 `SHOTS`번 측정해 모든 항의 기댓값을 한 번에 얻는다(항 수만큼 돌리지 않는다). 바인딩된 회로 2N+1개를 리스트로 보내는 것과 실행 내용이 같고, 회로를 한 번만 transpile해 두고 값만 바꿔 보내므로 전송량이 작다는 점만 다르다. 마지막 업데이트 뒤에는 최종 θ의 에너지만 재는 회로 1개짜리 job을 하나 더 보낸다.

| 방식 | step 수 | job당 회로 | 총 job | 총 회로 |
|---|---|---|---|---|
| PADO | 5 | 2·61 + 1 = 123 | 5 + 1 | 5·123 + 1 = 616 |
| standard | 20 | 2·114 + 1 = 229 | 20 + 1 | 20·229 + 1 = 4,581 |

매 step의 θ는 노이즈 없는 statevector로도 평가해(`energy_noiseless`) 하드웨어 노이즈와 최적화 진행을 나눠 볼 수 있게 한다.

### 4-1. Estimator 설정

In [8]:
estimator = EstimatorV2(mode=backend)          # mode=backend: job 모드 (step마다 job 하나)
estimator.options.default_shots = SHOTS        # 회로(행) 하나당 측정 횟수
if BACKEND_MODE == "ibm":
    estimator.options.resilience_level = RESILIENCE_LEVEL   # 실기기에서만 의미가 있다
    if REP_DELAY is not None:
        estimator.options.execution.rep_delay = REP_DELAY   # shot 사이 대기 (기기 허용 범위 안)

### 4-2. job 하나의 구성 보기

PADO 방식의 첫 step에서 job 하나에 들어갈 파라미터 행렬을 만들어 앞부분만 출력해 본다. 아무것도 보내지 않는다. 4-3의 학습 루프도 같은 방식으로 행렬을 만든다.

In [9]:
theta = theta0["pado_safe"]
k = len(theta)
rows = [theta.copy()]                      # 행 0: θ 그대로
for i in range(k):
    plus = theta.copy()
    plus[i] += np.pi / 2                   # 행 2i+1: θ_i만 +π/2
    minus = theta.copy()
    minus[i] -= np.pi / 2                  # 행 2i+2: θ_i만 −π/2
    rows += [plus, minus]
rows = np.array(rows)

print(f"rows 크기 = {rows.shape}  (2N+1 = {2 * k + 1}개 회로, N = {k}개 파라미터)")
for r in range(5):
    changed = np.flatnonzero(rows[r] != theta)
    if len(changed) == 0:
        print(f"  행 {r}: θ 그대로                → E(θ)")
    else:
        i = changed[0]
        sign = "+" if rows[r][i] > theta[i] else "−"
        print(f"  행 {r}: θ_{i}만 {sign}π/2 ({theta[i]:+.4f} → {rows[r][i]:+.4f}) → E_{i}{sign}")
print("  ...")
print(f"이 행렬, ISA 회로 1개, 관측량 H를 묶어 job 1개로 보낸다. 행마다 {SHOTS} shots씩 실행된다.")

rows 크기 = (123, 61)  (2N+1 = 123개 회로, N = 61개 파라미터)
  행 0: θ 그대로                → E(θ)
  행 1: θ_0만 +π/2 (+0.3263 → +1.8971) → E_0+
  행 2: θ_0만 −π/2 (+0.3263 → -1.2445) → E_0−
  행 3: θ_1만 +π/2 (-1.3019 → +0.2689) → E_1+
  행 4: θ_1만 −π/2 (-1.3019 → -2.8727) → E_1−
  ...
이 행렬, ISA 회로 1개, 관측량 H를 묶어 job 1개로 보낸다. 행마다 1024 shots씩 실행된다.


### 4-3. 학습 루프

방식마다 아래를 반복한다. 결과 파일 `params.json`은 step마다 덮어 써서 진행 상황을 남긴다. 이미 끝난 방식(최종 에너지까지 저장됨)은 건너뛴다.

1. 파라미터 행렬 `rows`를 만든다 (4-2와 같음).
2. job을 보내고 job ID를 먼저 저장한다. 실기기에서 기다리다 끊겨도 다음 실행 때 이 ID로 결과를 받아 온다.
3. 결과에서 E(θ)와 기울기 g_i = (E_i+ − E_i−)/2를 꺼낸다. IBM이 집계한 사용 시간(usage)과 시각도 같이 기록한다.
4. 같은 θ의 노이즈 없는 에너지를 statevector로 계산해 같이 기록한다.
5. Adam으로 θ를 업데이트하고 저장한다. Adam 식은 고전 실험의 `torch.optim.Adam`(β1 = 0.9, β2 = 0.999, ε = 1e-8)과 같다.

업데이트를 `STEPS[방식]`번 한 뒤 마지막으로 최종 θ의 에너지만 재는 job(회로 1개)을 보낸다.

In [10]:
for run in ["pado_safe", "standard"]:
    out = OUT_ROOT / run
    (out / "qasm").mkdir(parents=True, exist_ok=True)
    path = out / "params.json"

    # ---- 이어서 돌리기: 끝난 방식은 건너뛰고, 중간에 멈춘 방식은 저장된 θ와 Adam 상태에서 다시 시작
    if path.exists():
        rec = json.loads(path.read_text())
        if rec["final"] is not None:
            print(f"{run}: 끝난 실행이 있어 건너뜀 (결과는 덮어쓰지 않는다)")
            continue
    else:
        rec = {"run": run, "status": "running", "experiment": cfg["name"], "problem": CASE, "backend": backend.name,
               "e_min": e_min, "e_max": e_max,
               "config": {"shots": SHOTS, "resilience_level": RESILIENCE_LEVEL if BACKEND_MODE == "ibm" else None,
                          "rep_delay_s": rep, "steps": STEPS[run], "lr": LR, "optimization_level": OPT_LEVEL,
                          "seed_transpiler": SEED_TRANSPILER, "n_params": len(theta0[run]),
                          "circuits_per_step": 2 * len(theta0[run]) + 1, "physical_qubits": phys[run]},
               "theta_start": theta0[run].tolist(), "theta": theta0[run].tolist(),
               "adam_m": [0.0] * len(theta0[run]), "adam_v": [0.0] * len(theta0[run]),
               "steps": [], "pending_job": None, "final": None}
    theta = np.array(rec["theta"])      # 현재 파라미터
    m = np.array(rec["adam_m"])         # Adam 1차 모멘트 (기울기의 이동 평균)
    v = np.array(rec["adam_v"])         # Adam 2차 모멘트 (기울기 제곱의 이동 평균)
    k = len(theta)
    print(f"[{run}] 학습 시작: {STEPS[run]} step, step당 회로 {2 * k + 1}개 × {SHOTS} shots", flush=True)

    # t = 0 .. STEPS-1: parameter-shift job + 업데이트,  t = STEPS: 최종 에너지 job
    for t in range(len(rec["steps"]), STEPS[run] + 1):
        final = t == STEPS[run]

        # ---- 1. 이번 job에 넣을 파라미터 행렬
        if final:
            rows = np.array([theta])                 # 최종 에너지: θ 하나 → 회로 1개
        else:
            rows = [theta.copy()]                    # 행 0: θ 그대로 → E(θ)
            for i in range(k):
                plus = theta.copy()
                plus[i] += np.pi / 2                 # 행 2i+1: θ_i만 +π/2 → E_i+
                minus = theta.copy()
                minus[i] -= np.pi / 2                # 행 2i+2: θ_i만 −π/2 → E_i−
                rows += [plus, minus]
            rows = np.array(rows)                    # (2k+1) × k

        # ---- 2. job 보내기 (회로 1개 + 관측량 + 행렬 = PUB 1개 = job 1개). job ID를 먼저 저장
        t0 = time.perf_counter()
        if rec["pending_job"] is not None and BACKEND_MODE == "ibm":
            job = service.job(rec["pending_job"])   # 지난번에 보내 놓고 못 받은 job
        else:
            job = estimator.run([(isa[run], isa_H[run], rows)])
            rec["pending_job"] = job.job_id()
            save_json(path, rec)

        # ---- 3. 결과: 행마다 기댓값 하나 (evs[r] = 행 r의 에너지, stds[r] = 그 표준오차). IBM 사용 시간과 시각도 기록
        res = job.result()[0]
        evs = np.asarray(res.data.evs)
        stds = np.asarray(res.data.stds)
        energy = float(evs[0])                       # 행 0 = 이번 θ의 에너지
        usage_s, timestamps = None, {}
        try:
            usage_s = float(job.usage())
        except Exception:
            pass
        try:
            timestamps = {kk: str(vv) for kk, vv in job.metrics().get("timestamps", {}).items()}
        except Exception:
            pass

        # ---- 4. 같은 θ의 노이즈 없는 에너지 (비교용, 학습에는 쓰지 않음)
        e_noiseless = float(StatevectorEstimator().run([(circuits[run], H, theta)]).result()[0].data.evs)

        row = {"t": t, "job_id": job.job_id(), "theta": theta.tolist(), "energy": energy,
               "energy_std": float(stds[0]), "ar": ar(energy),
               "energy_noiseless": e_noiseless, "ar_noiseless": ar(e_noiseless), "n_circuits": len(rows),
               "wall_s": time.perf_counter() - t0, "usage_s": usage_s, "timestamps": timestamps}

        if final:
            rec["final"] = row
            rec["status"] = "ok"
        else:
            # ---- 5. 기울기: g_i = (E_i+ − E_i−) / 2
            grad = np.zeros(k)
            for i in range(k):
                e_plus = evs[2 * i + 1]
                e_minus = evs[2 * i + 2]
                grad[i] = (e_plus - e_minus) / 2
            row["grad"] = grad.tolist()
            rec["steps"].append(row)

            # ---- 6. Adam 업데이트 (torch.optim.Adam 기본값과 같은 식)
            step = t + 1
            m = 0.9 * m + 0.1 * grad
            v = 0.999 * v + 0.001 * grad ** 2
            m_hat = m / (1 - 0.9 ** step)            # 초반 편향 보정
            v_hat = v / (1 - 0.999 ** step)
            theta = theta - LR * m_hat / (np.sqrt(v_hat) + 1e-8)
            rec["theta"], rec["adam_m"], rec["adam_v"] = theta.tolist(), m.tolist(), v.tolist()

        # ---- 7. 저장 (step마다)
        rec["pending_job"] = None
        save_json(path, rec)
        print(f"[{run}] t={t:2d}{' (최종 에너지)' if final else ''}  AR={row['ar']:.4f} (noiseless {row['ar_noiseless']:.4f})  "
              f"{len(rows)} circuits  {row['wall_s']:.1f}s  usage {usage_s}  job {row['job_id']}", flush=True)

    # 시작·최종 θ를 대입한 회로를 OpenQASM으로 저장 (다른 하드웨어에서 다시 돌릴 때 사용)
    save_qasm(out / "qasm" / f"{run}_start.qasm", n, gates[run], theta0[run])
    save_qasm(out / "qasm" / f"{run}_final.qasm", n, gates[run], theta)
    save_json(out / "circuit.json", {"n_qubits": n, "gates": gates[run], "edges": circ_json["edges"],
                                     "fields": circ_json["fields"], "convention": circ_json["convention"]})

[pado_safe] 학습 시작: 5 step, step당 회로 123개 × 1024 shots


[pado_safe] t= 0  AR=0.8912 (noiseless 0.9311)  123 circuits  5.3s  usage 0.0  job fe984264-3560-4260-b21f-a5150d8ce771


[pado_safe] t= 1  AR=0.9123 (noiseless 0.9455)  123 circuits  5.5s  usage 0.0  job 53b7373c-306a-466e-9a2a-a7430788969a


[pado_safe] t= 2  AR=0.9135 (noiseless 0.9563)  123 circuits  5.4s  usage 0.0  job 91cda5fe-08e4-4e06-ac96-71e922717e80


[pado_safe] t= 3  AR=0.9212 (noiseless 0.9641)  123 circuits  9.3s  usage 0.0  job 8a3000c2-1fdf-4ed6-bdad-3eb881597ead


...

[standard] t=16  AR=0.7362 (noiseless 0.7928)  229 circuits  15.0s  usage 0.0  job 42fe597d-5dab-4972-a62a-b93aa2576a7f


[standard] t=17  AR=0.7383 (noiseless 0.8002)  229 circuits  14.8s  usage 0.0  job 999fc35c-80b9-4ff8-8217-5655c9c229c2


[standard] t=18  AR=0.7426 (noiseless 0.8069)  229 circuits  14.9s  usage 0.0  job 7184efac-e94b-4b35-b7c5-090d40edfb58


[standard] t=19  AR=0.7539 (noiseless 0.8129)  229 circuits  14.8s  usage 0.0  job 289aacd7-aa61-49ab-8e87-9fbf3cdce433


[standard] t=20 (최종 에너지)  AR=0.7478 (noiseless 0.8183)  1 circuits  3.5s  usage 0.0  job 63dc92da-a9b0-40e4-8353-ef11f139add6


## 5. PADO 방식의 증류 전 측정 (비교용 1회)

PADO 방식의 양자컴퓨터 학습은 증류된 회로에서 시작하므로, 사전학습 직후·증류 전 상태는 위 학습에서 재지 않는다. 증류가 AR을 얼마나 바꾸는지 보려고, LWPP 사전학습 직후의 θ(`theta_lwpp_final`, 각도 114개)를 증류 전 전체 회로(standard와 같은 ISA 회로)에 넣어 회로 1개짜리 job을 한 번 보낸다. 이 측정은 비교용이라 PADO 방식의 양자 자원(회로 수)에는 넣지 않는다. 결과는 `pado_safe/pre_distill.json`에 저장하고, 이미 있으면 다시 재지 않는다.

In [11]:
pre_path = OUT_ROOT / "pado_safe" / "pre_distill.json"
theta_pre = np.array(start["theta_lwpp_final"])          # LWPP 사전학습 직후, 증류 전 (전체 114개 각도)
if pre_path.exists():
    pre = json.loads(pre_path.read_text())
    print("pre_distill.json found, not measured again")
else:
    job = estimator.run([(isa["standard"], isa_H["standard"], theta_pre[None, :])])   # 전체 회로, 회로 1개
    res = job.result()[0]
    energy = float(np.asarray(res.data.evs)[0])
    e_noiseless = float(StatevectorEstimator().run([(circuits["standard"], H, theta_pre)]).result()[0].data.evs)
    pre = {"what": "PADO after LWPP pre-training, before distillation (theta_lwpp_final on the full circuit)",
           "backend": backend.name, "shots": SHOTS, "job_id": job.job_id(), "theta": theta_pre.tolist(),
           "energy": energy, "energy_std": float(np.asarray(res.data.stds)[0]), "ar": ar(energy),
           "energy_noiseless": e_noiseless, "ar_noiseless": ar(e_noiseless), "n_circuits": 1}
    save_json(pre_path, pre)
# 확인: 노이즈 없는 값은 시작점 파일의 LWPP 직후 에너지(PennyLane)와 같아야 한다
assert abs(pre["energy_noiseless"] - start["exact_energy"]["lwpp_final"]) < 1e-8
print(f"PADO before distillation: measured AR={pre['ar']:.4f}  noiseless AR={pre['ar_noiseless']:.4f}  job {pre['job_id']}")

PADO before distillation: measured AR=0.8243  noiseless AR=0.9210  job 6dfe5615-f3b5-452a-a987-3bc723f2ce6e


## 6. 끝

여기까지가 양자컴퓨터 실행이다. 회로 수·shots 합계, AR 처음/끝, 판정과 그림은 `python metric.py --exp TC3`가 `results_on_<backend>/TC3/metrics/`에 만든다.

In [12]:
for run in ["pado_safe", "standard"]:
    rec = json.loads((OUT_ROOT / run / "params.json").read_text())
    pts = rec["steps"] + ([rec["final"]] if rec["final"] is not None else [])
    print(f"{run:10s} step {len(rec['steps'])}/{STEPS[run]}  회로 {sum(p['n_circuits'] for p in pts)}  "
          f"AR {pts[0]['ar']:.4f} → {pts[-1]['ar']:.4f}  (noiseless {pts[0]['ar_noiseless']:.4f} → {pts[-1]['ar_noiseless']:.4f})")
print(f"다음: python metric.py --exp TC3   (지표·판정·그림 → results_on_{backend.name}/TC3/metrics/)")

pado_safe  step 5/5  회로 616  AR 0.8912 → 0.9303  (noiseless 0.9311 → 0.9729)
standard   step 20/20  회로 4581  AR 0.5323 → 0.7478  (noiseless 0.5291 → 0.8183)
다음: python metric.py --exp TC3   (지표·판정·그림 → results_on_fake_nighthawk/TC3/metrics/)
